<a href="https://colab.research.google.com/github/gautamkr1876/AIML_ClassNotes/blob/main/8.%20Agentic%20AI%20Systems/10.%20%20LlamaIndex%20%26%20Advanced%20RAG%20Frameworks/L9_LlamaIndex_Advanced_RAG.ipynb" target="_blank">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# LlamaIndex & Advanced RAG Frameworks

Welcome to today's session. By the end of this class, you will be able to:

1. Explain why **LlamaIndex** exists alongside LangChain and when each shines.
2. Build and query four different index types: **Vector, Tree, Keyword, and Knowledge Graph**.
3. Use the **Sub-Question Query Engine** to answer complex, cross-document questions.
4. Add **persistent memory** to agents using **Agno** (the evolution of Phidata).
5. Make an informed framework choice for your own RAG projects.

**Stack we will use today:** `llama-index` + **Groq** (LLM) + **HuggingFace** (embeddings) + **Agno** + **Wikipedia**.

Keep the chat window open. We will use chat reactions, quick polls, and 5 MCQs to stay in sync.

In [ ]:
# Install all dependencies for the full notebook.
# (Run once, then Runtime -> Restart session before running anything else.)
#
# IMPORTANT: do NOT pin llama-index down to 0.11.x. Colab ships a newer
# llama-index-core, and forcing an old pin over it leaves a MIXED install
# (the "cannot import name 'ImageBlock'" error: new modules + old __init__).
# We install a coherent 0.12-line set instead -> ImageBlock exists everywhere,
# and every index type this notebook uses is still present.
#
# Embeddings come from FastEmbed (ONNX): no torch / sentence-transformers /
# huggingface-hub[inference], so no resolver thrash and no source builds.
#
# We DON'T install llama-index-question-gen-openai: it drags in the
# agent/workflows stack (the "cannot import name 'StateStore'" mismatch) and
# we don't need it -- the core LLMQuestionGenerator handles sub-questions.
#
# Agno (Segment 7) is PINNED to the 2.x line. The agent API used there
# (db=SqliteDb, enable_user_memories, add_history_to_context) is the Agno 2.x
# API; an unpinned install that floats to a future major version would break
# those cells -- which is exactly what went wrong before.

%pip install -q -U pip

# Pass 1: a self-consistent LlamaIndex 0.12 stack. -U so any older/mixed core
# already in the runtime is fully replaced (not overlaid).
%pip install -q --prefer-binary -U \
    "llama-index-core>=0.12,<0.13" \
    "llama-index-llms-groq>=0.3,<0.4" \
    "llama-index-embeddings-fastembed>=0.3,<0.4" \
    "llama-index-readers-file>=0.4,<0.5" \
    "wikipedia==1.4.0" \
    "networkx==3.3" \
    "pyvis==0.3.2"

# Pass 2: Agno (agent runtime + memory) on its own pass, so its deps don't
# fight the pins above. The [sqlite] extra brings SQLAlchemy for SqliteDb;
# `groq` is Agno's client for the Groq model.
%pip install -q --prefer-binary "agno[sqlite]>=2.6,<3" "groq"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.11.0+cpu requires setuptools<82, but you have setuptools 82.0.1 which is incompatible.


In [ ]:
import os

# Removed: %pip install -U numpy # Upgrade numpy to resolve potential version conflicts
# This caused an incompatibility with llama-index-core, which requires numpy<2.0.0.

# Groq is free-tier friendly. Get a key at https://console.groq.com
#
# In Google Colab: click the key icon (🔑) in the left sidebar, add a secret
# named GROQ_API_KEY, paste your key, and toggle "Notebook access" ON.
# This cell then reads it automatically, no prompt needed.
if "GROQ_API_KEY" not in os.environ:
    try:
        from google.colab import userdata
        os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
        print("Loaded GROQ_API_KEY from Colab secrets.")
    except Exception as e:
        # Not on Colab, or the secret is missing / access not granted.
        from getpass import getpass
        os.environ["GROQ_API_KEY"] = getpass("Enter your GROQ_API_KEY: ")

from llama_index.core import Settings
from llama_index.llms.groq import Groq
from llama_index.embeddings.fastembed import FastEmbedEmbedding

# One-time global config. Every index built below will inherit these.
Settings.llm = Groq(model="llama-3.1-8b-instant", temperature=0.1)

# Embeddings via FastEmbed (ONNX): same model as before, but no torch /
# sentence-transformers, so it installs and loads cleanly on Colab.
# First call downloads the small bge-small model (~130 MB) and caches it.
Settings.embed_model = FastEmbedEmbedding(model_name="BAAI/bge-small-en-v1.5")
Settings.chunk_size = 512
Settings.chunk_overlap = 50

print("LLM:", Settings.llm.model)
print("Embeddings:", getattr(Settings.embed_model, "model_name", "BAAI/bge-small-en-v1.5"))
print("Setup complete. We are ready to begin.")

Loaded GROQ_API_KEY from Colab secrets.


/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'validate_default' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'validate_default' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authen

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/706 [00:00<?, ?B/s]

model_optimized.onnx:   0%|          | 0.00/66.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

LLM: llama-3.1-8b-instant
Embeddings: BAAI/bge-small-en-v1.5
Setup complete. We are ready to begin.


## Segment 1: Opening & Recap (0:00 - 0:10)

**[Instructor, 0:00]**

Welcome in, everyone. Drop a **"1" in chat** if you can hear me clearly and the notebook is loaded on your end. I will wait 20 seconds for the chat to fill up.

Quick warm-up question, type your answer in chat in one or two words:

> *"In your last LangChain session, what was the single most painful part of building a RAG pipeline?"*

Common answers I see: "chunking", "retrieval quality", "too many wrappers", "debugging chains". Hold that thought, because today's framework was literally born to fix some of these.

**[Instructor, 0:03]**

Let us quickly align on where we are. You already know:

- **RAG** = Retrieve relevant chunks, then Augment the prompt, then Generate.
- **LangChain** treats your problem as a **chain of operations**. It is LLM-centric. Think of LangChain as a Swiss Army knife for LLM workflows: agents, tools, chains, memory, prompts, all bolted together.

Now here is the thing. When **Jerry Liu** started LlamaIndex in late 2022, he noticed something. Most teams were not struggling with the LLM call itself. They were struggling with **the data**. How do you ingest 50,000 PDFs? How do you query a knowledge graph and a vector store in the same breath? How do you summarize a 400-page annual report without blowing the context window?

So LlamaIndex took a different stance. It is **data-centric**, not LLM-centric. The mental model is:

> *"Your data is the product. The LLM is just the interface."*

**[Instructor, 0:07]**

Think of it this way. **LangChain is like building with Lego blocks.** Every block is a chain, a tool, a prompt. You snap them together. Powerful, but you are the architect.

**LlamaIndex is like IKEA furniture for data.** It already knows the patterns: "you want to summarize a long doc? Use a Tree Index. You want entity relationships? Knowledge Graph Index. You want hybrid search? Already built in."

Both are excellent. The choice depends on the job. We will come back to this comparison at the end of class.

**Chat prompt before we move on:**

> *Type in chat: "data-first" or "LLM-first", which describes your current project?*

Take 30 seconds, then we dive into code.

## Segment 2: Vector Index (0:10 - 0:30)

**[Instructor, 0:10]**

Let us start with the index type you are already 80% familiar with: the **Vector Index**.

![image](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/207/579/original/img1.png?1782722196)

**The Netflix analogy.** When you finish watching *Stranger Things*, Netflix does not look up a SQL table called "horror_sci_fi_80s_kids". It converts every show into a **vector**, a list of numbers that captures its "vibe": eerie, nostalgic, ensemble cast, supernatural. Then it finds shows whose vectors are **mathematically close** to *Stranger Things*. That is semantic similarity, and that is exactly what a Vector Index does for your documents.

When you ask *"What is our refund policy?"*, the Vector Index does not search for the word "refund". It searches for **chunks whose meaning is closest** to your question. The chunk could say "money back guarantee within 30 days" and still get retrieved. Words differ, meaning matches.

Mathematically, similarity between a query vector $q$ and a chunk vector $c$ is most commonly cosine similarity:

$$\text{sim}(q, c) = \frac{q \cdot c}{\|q\| \, \|c\|}$$

**Quick chat reaction:** Thumbs up react if the Netflix analogy clicks for you.

Let us build one.

In [ ]:
# First, we generate some synthetic company documents.
# Imagine these are internal docs at a fictional company called "NimbusCloud".

import os
import shutil

DATA_DIR = "nimbus_docs"
if os.path.exists(DATA_DIR):
    shutil.rmtree(DATA_DIR)
os.makedirs(DATA_DIR)

docs = {
    "hr_policy.txt": """
    NimbusCloud HR Policy Handbook (2024).
    Employees are entitled to 24 days of paid leave annually.
    Maternity leave is 26 weeks, paternity leave is 4 weeks.
    Sick leave does not require a doctor's note for up to 2 consecutive days.
    All employees must complete annual compliance training by December 31.
    Remote work is permitted up to 3 days per week with manager approval.
    """,
    "refund_policy.txt": """
    NimbusCloud Customer Refund Policy.
    Customers may request a full refund within 30 days of purchase.
    Subscription cancellations after 30 days are prorated.
    Enterprise contracts have separate refund terms negotiated per agreement.
    Refunds are processed within 7 business days to the original payment method.
    No refunds are issued for usage-based overage charges already consumed.
    """,
    "security_policy.txt": """
    NimbusCloud Security and Data Handling Policy.
    All customer data is encrypted at rest using AES-256.
    Data in transit uses TLS 1.3.
    Employees must use multi-factor authentication for all internal systems.
    Production database access is restricted to the SRE team.
    Security incidents must be reported to the CISO within 1 hour of detection.
    """,
    "product_roadmap.txt": """
    NimbusCloud Product Roadmap Q1-Q2 2025.
    Q1: Launch multi-region failover for the storage product.
    Q1: Introduce GPU-backed compute instances for AI workloads.
    Q2: Release fine-grained IAM policies and audit log streaming.
    Q2: Beta launch of the serverless inference endpoint product.
    The roadmap is subject to change based on customer feedback.
    """,
}

for fname, content in docs.items():
    with open(os.path.join(DATA_DIR, fname), "w") as f:
        f.write(content.strip())

print(f"Created {len(docs)} synthetic documents in '{DATA_DIR}/'")
print("Files:", os.listdir(DATA_DIR))

Created 4 synthetic documents in 'nimbus_docs/'
Files: ['hr_policy.txt', 'refund_policy.txt', 'security_policy.txt', 'product_roadmap.txt']


In [ ]:
# Load the documents and build a Vector Index
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex

documents = SimpleDirectoryReader(DATA_DIR).load_data()
print(f"Loaded {len(documents)} documents.")

vector_index = VectorStoreIndex.from_documents(documents)
print("Vector index built. Embeddings computed for all chunks.")

Loaded 4 documents.
Vector index built. Embeddings computed for all chunks.


![image](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/207/580/original/img2.png?1782722319)

In [ ]:
# Query the Vector Index using natural language
vector_query_engine = vector_index.as_query_engine(similarity_top_k=2)

response = vector_query_engine.query("How many days of leave do employees get?")
print("Q: How many days of leave do employees get?")
print("A:", response)
print("\n--- Source chunks used ---")
for node in response.source_nodes:
    print(f"[{node.metadata.get('file_name')}] score={node.score:.3f}")

Q: How many days of leave do employees get?
A: Employees are entitled to 24 days of paid leave annually.

--- Source chunks used ---
[hr_policy.txt] score=0.778
[refund_policy.txt] score=0.570


In [ ]:
# Try a semantic question where the keyword does NOT appear in the docs
response = vector_query_engine.query("Can I get my money back if I cancel early?")
print("Q: Can I get my money back if I cancel early?")
print("A:", response)
print("\nNotice: the docs say 'refund', not 'money back'. Semantic search still finds it.")

Q: Can I get my money back if I cancel early?
A: Yes, you can get a full refund if you cancel within a certain timeframe.

Notice: the docs say 'refund', not 'money back'. Semantic search still finds it.


**[Instructor, 0:25]**

See what happened in that last query? The document says **"refund"**, the user asked about **"money back"**. A keyword search would have failed. Vector search nailed it because the embeddings live in meaning-space, not word-space.

**Chat prompt:** Type in chat one example from YOUR project where vector search would beat keyword search. I will read 2-3 out loud.

**When to use Vector Index:**
- Semantic Q&A over a document corpus.
- "Find similar" workflows (similar tickets, similar resumes, similar bugs).
- The default choice for most RAG pipelines. Start here.

**When NOT to use it alone:**
- Summarizing a whole document (vector retrieval gives you only top-k chunks).
- Questions requiring strict keyword/identifier matching ("show me the doc with serial number XJ-9921").
- Multi-hop reasoning across entities.

For those, we have other tools. But first, an MCQ.

### MCQ 1 of 5

**Question:** A user types the query *"Can I work from home twice a week?"* into a RAG app built on a Vector Index over NimbusCloud's HR docs. The HR policy document literally says *"Remote work is permitted up to 3 days per week."*. Which of the following is the MOST accurate explanation of why the Vector Index can still answer correctly?

**Options:**

- **A.** The Vector Index performs fuzzy string matching on "work from home" and "remote work".
- **B.** The Vector Index converts both the query and the document chunks into embeddings, and retrieves chunks whose semantic meaning is close to the query, regardless of exact wording.
- **C.** The Vector Index sends the entire HR policy to the LLM, which then figures out the answer on its own.
- **D.** The Vector Index uses a synonym dictionary to expand "work from home" into "remote work" before searching.

*React in chat with A, B, C, or D.*

**Correct answer: B.** Vector indices operate in embedding space. Both the query and document chunks are mapped to high-dimensional vectors, and similarity is measured geometrically (cosine similarity, typically). No fuzzy matching, no synonym dictionary, no full-document stuffing.

## Segment 3: Tree Index and Keyword Index (0:30 - 0:50)

**[Instructor, 0:30]**

Now, Vector Index is great for "needle in a haystack" questions. But what if your question is *"Summarize the entire HR handbook"*? Top-k retrieval will give you 3 chunks out of 50. You will get a partial answer.

![image](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/207/581/original/img3.png?1782722360)

Enter the **Tree Index**.

**The Wikipedia TOC analogy.** Open any long Wikipedia article, say "World War II". You see a Table of Contents on the left: Background, European Theater, Pacific Theater, Aftermath. Each section has subsections. To get a "summary of the war", Wikipedia editors do not paste the whole article. They wrote a lead paragraph that summarizes the sections, which themselves summarize the subsections.

A **Tree Index** does this automatically. It chunks your docs, summarizes groups of chunks into parent nodes, then summarizes those parents into grandparents, all the way up to one root node. When you query, it traverses **top-down**: read the root, decide which child branch is relevant, recurse.

**Use it when:** you need **summarization** or **hierarchical reasoning** over large documents.

**The Keyword Index** is the opposite extreme. It is like the **back-of-the-book index** in a textbook, or a library card catalog. LlamaIndex uses the LLM to extract keywords from each chunk and from your query, then matches them. Fast, cheap, deterministic. Useful when you need **exact term matching** (product codes, names, acronyms) and embeddings would over-generalize.

In [ ]:
# Build a Tree Index over the same documents
from llama_index.core import TreeIndex

tree_index = TreeIndex.from_documents(documents, num_children=3)
print("Tree index built. The LLM summarized chunks bottom-up.")

Tree index built. The LLM summarized chunks bottom-up.


In [ ]:
# Tree Index excels at SUMMARIZATION queries
tree_query_engine = tree_index.as_query_engine()
response = tree_query_engine.query(
    "Give me a one-paragraph executive summary of NimbusCloud's policies and roadmap."
)
print("Q: Executive summary across all docs")
print("A:", response)

Q: Executive summary across all docs
A: NimbusCloud is committed to providing a supportive and flexible work environment for its employees. Key initiatives include offering generous paid leave policies, with 24 days of annual leave and comprehensive maternity and paternity leave options. Additionally, the company prioritizes employee well-being by allowing remote work arrangements and providing a reasonable sick leave policy. Furthermore, NimbusCloud emphasizes the importance of ongoing professional development through annual compliance training, ensuring employees stay up-to-date on the latest best practices.


In [ ]:
# Build a Keyword Index over the same docs
from llama_index.core import SimpleKeywordTableIndex

keyword_index = SimpleKeywordTableIndex.from_documents(documents)
print("Keyword index built. Keywords extracted per chunk.")

Keyword index built. Keywords extracted per chunk.


In [ ]:
# Keyword Index shines on exact-term lookups
keyword_query_engine = keyword_index.as_query_engine()
response = keyword_query_engine.query("What does the policy say about AES-256?")
print("Q: What does the policy say about AES-256?")
print("A:", response)

Q: What does the policy say about AES-256?
A: Empty Response


**[Instructor, 0:45]**

Quick recap of what we just saw:

| Index Type | Best at | Cost | Analogy |
|------------|---------|------|---------|
| Vector | Semantic Q&A, "find similar" | Embedding compute | Netflix recommendations |
| Tree | Summarization, hierarchical Q&A | More LLM calls upfront | Wikipedia TOC |
| Keyword | Exact-term lookup | Cheap, no embeddings needed | Book back-index |

**Chat prompt:** Type in chat which of these three you have NEVER used before. I will spend extra time on whichever wins the poll.

### MCQ 2 of 5

**Question:** Your team needs to build a Q&A bot over a single 300-page **annual report**. The most common user question is: *"Summarize the company's financial performance and outlook for the next year."*. Which index type is the BEST fit as the primary index?

**Options:**

- **A.** Vector Index, because it has the lowest query latency.
- **B.** Keyword Index, because financial reports contain lots of specific numbers and terms.
- **C.** Tree Index, because it builds hierarchical summaries that can answer broad summarization queries without truncation.
- **D.** No index at all. Just send the whole 300-page report to the LLM in the context window.

*React with A, B, C, or D.*

**Correct answer: C.** Vector Indices return only top-k chunks, which is insufficient for full-document summarization. Keyword Index is great for exact terms but not narrative summaries. Stuffing 300 pages exceeds most context windows and is expensive even when it fits. Tree Index was designed exactly for this use case.

## Segment 4: Knowledge Graph Index (0:50 - 1:10)

**[Instructor, 0:50]**

Now we step into territory where Vector Indices genuinely cannot help: **multi-hop relationship questions**.

![image](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/207/582/original/img4.png?1782722408)

**The Google Knowledge Panel analogy.** Google "Elon Musk" and you see a panel on the right: born 1971, founded SpaceX, CEO of Tesla, owns X. Now ask *"Who founded the company that makes the Cybertruck?"*. Google does not vector-search the web. It walks a **knowledge graph**: Cybertruck -> made_by -> Tesla -> founded_by -> Martin Eberhard, Marc Tarpenning (and Elon Musk joined later as chairman/CEO).

A **Knowledge Graph Index** extracts triplets `(subject, relation, object)` from your text and builds an actual graph. Queries traverse edges. This is brilliant for:

- "Who reports to whom?" (org charts)
- "Which products depend on which services?" (architecture docs)
- "What's the chain of causation in this incident report?"

Let us build one over a real Wikipedia article.

In [ ]:
# Fetch a Wikipedia article. We'll use Tesla, Inc. for the rich entity relationships.
import wikipedia
from llama_index.core import Document

try:
    wiki_page = wikipedia.page("Tesla, Inc.", auto_suggest=False)
    tesla_text = wiki_page.content[:8000]  # keep it manageable for class demo
    print(f"Fetched Tesla, Inc. article. Length: {len(tesla_text)} chars (truncated).")
except Exception as e:
    print(f"Wikipedia fetch failed ({e}). Falling back to a small synthetic blurb.")
    tesla_text = (
        "Tesla, Inc. is an American electric vehicle and clean energy company "
        "headquartered in Austin, Texas. Tesla was founded in 2003 by Martin "
        "Eberhard and Marc Tarpenning. Elon Musk joined Tesla in 2004 as chairman "
        "and later became CEO. Tesla produces the Model S, Model 3, Model X, "
        "Model Y, and the Cybertruck. JB Straubel was a co-founder and former CTO."
    )

tesla_docs = [Document(text=tesla_text)]

Fetched Tesla, Inc. article. Length: 8000 chars (truncated).


In [ ]:
# Build the Knowledge Graph Index
from llama_index.core import KnowledgeGraphIndex
from llama_index.core.graph_stores import SimpleGraphStore
from llama_index.core import StorageContext

graph_store = SimpleGraphStore()
storage_context = StorageContext.from_defaults(graph_store=graph_store)

kg_index = KnowledgeGraphIndex.from_documents(
    tesla_docs,
    max_triplets_per_chunk=5,
    storage_context=storage_context,
    include_embeddings=True,
)
print("Knowledge Graph Index built.")
print("This took longer because the LLM had to extract entities and relations from each chunk.")

Knowledge Graph Index built.
This took longer because the LLM had to extract entities and relations from each chunk.


In [ ]:
# Peek at the extracted triplets
rel_map = graph_store.get_rel_map()
print("Sample triplets extracted from Tesla article:")
count = 0
for entity, relations in rel_map.items():
    for rel in relations[:2]:
        # rel is typically a list/tuple like [subject, relation, object]
        try:
            print(f"  {entity}  --[{rel[1]}]-->  {rel[2]}")
        except Exception:
            print(f"  {entity} -> {rel}")
        count += 1
        if count >= 10:
            break
    if count >= 10:
        break

Sample triplets extracted from Tesla article:
  Tesla  --[Is]-->  American multinational automotive and clean energy company
  Tesla  --[Headquartered in]-->  Austin
  Martin eberhard  --[Said he wanted to build]-->  Car manufacturer that is also a technology company
  Ian wright  --[Joined]-->  Martin eberhard and marc tarpenning
  Elon musk  --[Became]-->  Chairman of the board of directors
  Elon musk  --[Took]-->  Active role
  J. b. straubel  --[Joined]-->  Tesla as chief technical officer
  Solarcity  --[Purchased by]-->  Tesla
  Solarcity  --[Is]-->  American multinational automotive and clean energy company


In [ ]:
# Query the Knowledge Graph
kg_query_engine = kg_index.as_query_engine(
    include_text=True,
    response_mode="tree_summarize",
)

response = kg_query_engine.query("Who founded Tesla and when?")
print("Q: Who founded Tesla and when?")
print("A:", response)

Q: Who founded Tesla and when?
A: Martin Eberhard and Marc Tarpenning founded Tesla, and they incorporated the company on July 1, 2003.


In [ ]:
# A multi-hop question that benefits from graph traversal
response = kg_query_engine.query(
    "What products does Tesla make, and where is the company headquartered?"
)
print("Q: What products does Tesla make, and where is the company headquartered?")
print("A:", response)

Q: What products does Tesla make, and where is the company headquartered?
A: Tesla designs and manufactures battery electric vehicles, stationary battery energy storage devices, solar panels, solar shingles, and related products and services. The company is headquartered in Austin, Texas.


**[Instructor, 1:08]**

What you just saw is qualitatively different from Vector retrieval. The KG index understood that "Tesla" is an entity with multiple relations (`founded_by`, `headquartered_in`, `produces`) and walked those relations to assemble the answer.

**Real-world use:** LinkedIn's economic graph, Amazon's product knowledge graph, and most enterprise compliance systems are built on this exact pattern.

**Chat prompt:** React with a thumbs up if you can see a use case for KG in your current work. Type the use case in chat if you have one.

### MCQ 3 of 5

**Question:** Your fintech company has a corpus of regulatory documents where one question is: *"Which subsidiary of HoldingCo is regulated by which authority, and which executive signs off on compliance for each?"*. Which index choice gives the best results?

**Options:**

- **A.** Vector Index, increasing similarity_top_k to 20.
- **B.** Keyword Index, because regulator names are specific terms.
- **C.** Knowledge Graph Index, because the question requires traversing relationships between subsidiaries, regulators, and executives.
- **D.** Tree Index, because regulatory documents are long.

*React with your answer.*

**Correct answer: C.** This is a textbook multi-hop, multi-entity relationship question. Vector retrieval would grab chunks but might miss the connections. KG indices excel because they explicitly model `subsidiary -> regulated_by -> authority` and `subsidiary -> compliance_signoff -> executive` as graph edges.

## Segment 5: Break (1:10 - 1:15)

**[Instructor, 1:10]**

Five-minute breather. Stretch, water, deep breath.

While you stretch, drop one word in chat answering:

> *"Which index has surprised you the most so far: Vector, Tree, Keyword, or Knowledge Graph?"*

I will read out the top vote when we resume. See you at 1:15.

## Segment 6: Sub-Question Query Engine (1:15 - 1:35)

**[Instructor, 1:15]**

Welcome back. Now we tackle the most powerful pattern in LlamaIndex: the **Sub-Question Query Engine**.

![image](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/207/584/original/img5.png?1782722460)

**The Amazon analogy.** You type into Amazon: *"Best laptop under 80,000 rupees for video editing with at least 16GB RAM and a dedicated GPU."*. Amazon's search does not run that as one query. It decomposes it:

1. Filter category: laptops
2. Filter price: under 80,000
3. Filter use-case: video editing
4. Filter RAM: >= 16GB
5. Filter GPU: dedicated

Five sub-queries, run in parallel, results combined.

The **Sub-Question Query Engine** does this for documents. Given a complex question that spans multiple data sources, it:

1. Uses the LLM to **decompose** your question into sub-questions.
2. Routes each sub-question to the **appropriate underlying query engine** (one per document/source).
3. **Synthesizes** the sub-answers into a final coherent response.

This is how you build "ask anything across our entire company knowledge base" features.

In [ ]:
# Fetch 2 more Wikipedia articles for cross-document comparison
companies_to_fetch = ["Netflix", "OpenAI"]
company_docs = {}

for company in companies_to_fetch:
    try:
        page = wikipedia.page(company, auto_suggest=False)
        company_docs[company] = [Document(text=page.content[:8000])]
        print(f"Fetched {company} ({len(page.content[:8000])} chars)")
    except Exception as e:
        print(f"Fallback for {company}: {e}")
        company_docs[company] = [Document(text=f"{company} is a major technology company.")]

# Also reuse Tesla docs from earlier
company_docs["Tesla"] = tesla_docs
print("\nAll three company corpora ready.")

Fallback for Netflix: Expecting value: line 1 column 1 (char 0)
Fallback for OpenAI: Expecting value: line 1 column 1 (char 0)

All three company corpora ready.


In [ ]:
# Build a separate vector index per company, then expose each as a tool
from llama_index.core.tools import QueryEngineTool, ToolMetadata

company_engines = {}
for company, docs in company_docs.items():
    idx = VectorStoreIndex.from_documents(docs)
    company_engines[company] = idx.as_query_engine(similarity_top_k=2)

query_engine_tools = [
    QueryEngineTool(
        query_engine=company_engines["Tesla"],
        metadata=ToolMetadata(
            name="tesla_docs",
            description="Information about Tesla, Inc.: founding, products, headquarters, executives."
        ),
    ),
    QueryEngineTool(
        query_engine=company_engines["Netflix"],
        metadata=ToolMetadata(
            name="netflix_docs",
            description="Information about Netflix: founding, business model, content, executives."
        ),
    ),
    QueryEngineTool(
        query_engine=company_engines["OpenAI"],
        metadata=ToolMetadata(
            name="openai_docs",
            description="Information about OpenAI: founding, mission, products like ChatGPT and GPT-4, leadership."
        ),
    ),
]

print(f"Built {len(query_engine_tools)} query-engine tools.")

Built 3 query-engine tools.


In [ ]:
# Wire the tools into a SubQuestionQueryEngine.
# The question generator decomposes a complex query into per-tool sub-questions.
# We use the core LLM-based generator, which runs on Settings.llm (our Groq
# model) -- no OpenAI-specific package required (and that package is NOT
# installed, so importing it here would crash the notebook).
from llama_index.core.query_engine import SubQuestionQueryEngine
from llama_index.core.question_gen import LLMQuestionGenerator

question_gen = LLMQuestionGenerator.from_defaults(llm=Settings.llm)

sub_question_engine = SubQuestionQueryEngine.from_defaults(
    query_engine_tools=query_engine_tools,
    question_gen=question_gen,
    use_async=False,
    verbose=True,
)
print("Sub-Question Query Engine ready.")

Sub-Question Query Engine ready.


**[Instructor, 1:32]**

Watch the `verbose=True` output above. The engine literally printed each sub-question it generated, which tool it routed to, and the intermediate answer. This transparency is gold for debugging.

**Why this matters in production:** Imagine you work at a bank with separate document stores for: loan policies, KYC docs, compliance regulations, internal HR rules. A user asks: *"Am I allowed to approve a 5 crore loan to a related party?"*. That question touches three stores. Sub-Question Query Engine routes each sub-question correctly.

**Chat prompt:** Type "yes" or "no" in chat: have you ever built something that needed to query 2+ data sources and combine results?

### MCQ 4 of 5

**Question:** A SubQuestionQueryEngine receives the user query: *"How do Tesla's and Netflix's founding stories compare?"*. The engine has two tools: `tesla_docs` and `netflix_docs`. Which sequence BEST describes what happens?

**Options:**

- **A.** The engine sends the full query to both tools in parallel and concatenates the responses.
- **B.** The engine uses the LLM to break the query into sub-questions like "When was Tesla founded?" and "When was Netflix founded?", routes each to the appropriate tool, then synthesizes a comparison.
- **C.** The engine picks the tool whose description has the highest cosine similarity to the query and sends the full query only to that one.
- **D.** The engine merges both document corpora into a single vector store and runs one query against it.

*React with A, B, C, or D.*

**Correct answer: B.** The decomposition + routing + synthesis pattern is the defining behavior of SubQuestionQueryEngine. Option A loses precision, C ignores half the data, D defeats the purpose of having specialized tools per source.

## Segment 7: Agno for Agent Memory and Persistent Storage (1:35 - 1:50)

**[Instructor, 1:35]**

So far everything has been **stateless** RAG: each query is independent. Real agents need **memory** and they need it to **survive restarts**. This is where **Agno** comes in.

![image](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/207/585/original/img6.png?1782722491)

**Where Agno fits.** LlamaIndex is our *retrieval* layer: it indexes documents and answers questions over them. Agno is an *agent runtime*: it wraps an LLM with persistent state, memory, tools, and knowledge. They compose well: a LlamaIndex query engine can become a tool that an Agno agent calls.

**Quick history note.** You may have seen this framework called **Phidata** in older tutorials. The team rebranded and rebuilt it as **Agno** in 2024; the 2.x line we use here has a cleaner, db-centric API.

**The ChatGPT memory analogy.** When ChatGPT remembers that you are vegetarian, that you live in Bangalore, that you are learning Spanish, that is persistent memory: it is not in the current conversation, it lives across sessions.

Agno gives an agent a few distinct layers, and it is important not to confuse them:

1. **Session storage (short-term):** the running conversation. Lets a follow-up like *"based on my role..."* resolve correctly. Tied to a `session_id`.
2. **User memory (long-term):** durable facts the agent *distills* about a user ("Aanya is a data engineer"), reused across different sessions. Tied to a `user_id`.
3. **Knowledge:** static reference material the agent can search. This is basically RAG, the LlamaIndex half of today's class.

All of this is persisted to a single **SQLite file**, so it survives a process restart. Let us build a tutor agent and watch each layer work.

In [ ]:
# Agno 2.x: a persistent, stateful agent backed by a single SQLite file.
import os
from agno.agent import Agent
from agno.db.sqlite import SqliteDb
from agno.models.groq import Groq as AgnoGroq

# One SQLite file is the agent's entire persistent brain: session message
# history, long-term user memories, and more -- all of it survives a restart.
DB_FILE = "nimbus_agent.db"
if os.path.exists(DB_FILE):
    os.remove(DB_FILE)            # start clean so the demo is repeatable

agent_db = SqliteDb(db_file=DB_FILE)

USER_ID = "aanya"
SESSION_ID = "onboarding_chat_001"

tutor = Agent(
    name="NimbusTutor",
    # 70B here (not the 8B used elsewhere): Agno makes structured tool-style
    # calls under the hood to extract user memories, and the larger model is
    # noticeably more reliable at that. Swap to llama-3.1-8b-instant if you hit
    # rate limits -- the session-history layer works on either.
    model=AgnoGroq(id="llama-3.3-70b-versatile"),
    db=agent_db,
    description="You are a helpful AI tutor for the NimbusCloud onboarding program.",
    # Layer 1 -- session storage: feed prior turns of THIS session back to the LLM.
    add_history_to_context=True,
    num_history_runs=5,
    # Layer 2 -- long-term user memories: let Agno auto-extract durable facts.
    enable_user_memories=True,
    markdown=True,
)
print("Agno tutor agent created. Persistent brain ->", DB_FILE)

Agno tutor agent created. Persistent brain -> nimbus_agent.db


### Layer 1 — session memory in action

We talk to the agent across two turns. The second turn never repeats our role, but because the first turn is replayed into the context, the agent still knows what *"my role"* means.

In [ ]:
# Turn 1: introduce ourselves. Agno writes this exchange to the session
# history in SQLite, keyed by SESSION_ID.
r1 = tutor.run(
    "Hi! My name is Aanya and I just joined NimbusCloud as a data engineer.",
    user_id=USER_ID,
    session_id=SESSION_ID,
)
print(r1.content)

ERROR    Error calling Groq API: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust 
         your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code':         
         'tool_use_failed', 'failed_generation': '<function=add_memory={"memory": "Aanya works at NimbusCloud as a 
         data engineer", "topics": ["occupation", "location"]} </function>'}}

ERROR    Non-retryable model provider error: {"error":{"message":"Failed to call a function. Please adjust your    
         prompt. See 'failed_generation' for more                                                                  
         details.","type":"invalid_request_error","code":"tool_use_failed","failed_generation":"\u003cfunction=add_
         memory={\"memory\": \"Aanya works at NimbusCloud as a data engineer\", \"topics\": [\"occupation\",       
         \"location\"]} \u003c/function\u003e"}}

WARNING  Error in memory creation: {"error":{"message":"Failed to call a function. Please adjust your prompt. See  
         'failed_generation' for more                                                                              
         details.","type":"invalid_request_error","code":"tool_use_failed","failed_generation":"\u003cfunction=add_
         memory={\"memory\": \"Aanya works at NimbusCloud as a data engineer\", \"topics\": [\"occupation\",       
         \"location\"]} \u003c/function\u003e"}}

### Welcome to NimbusCloud, Aanya
It's great to have you on board as a data engineer. I'm here to help you with the onboarding process and answer any questions you may have. 

Since we just started, I don't have any prior knowledge or context about your specific role or requirements. However, I'll do my best to provide you with relevant information and guidance throughout the onboarding process.

To get started, is there something specific you'd like to know about NimbusCloud or your role as a data engineer? Are you looking for an overview of the company, information about your team, or perhaps details about the tools and technologies you'll be working with?


In [ ]:
# Turn 2: a follow-up that never mentions the role again. Because the prior
# turn is in context, "my role" still resolves to "data engineer".
r2 = tutor.run(
    "Based on my role, what should I learn first at NimbusCloud?",
    user_id=USER_ID,
    session_id=SESSION_ID,
)
print(r2.content)

### Prioritized Learning for Data Engineers at NimbusCloud
As a data engineer at NimbusCloud, it's essential to focus on the foundation of our data infrastructure and toolsets. Here are the key areas to learn first:

1. **Cloud Platform**: Familiarize yourself with our cloud provider (e.g., AWS, GCP, Azure) and its services, such as storage, computing, and networking.
2. **Data Architecture**: Understand our data architecture, including data lakes, warehouses, and pipelines. Learn about the different data sources, ingestion methods, and data processing workflows.
3. **Data Engineering Tools**: Learn about the tools and technologies used by our data engineering team, such as:
	* Apache Beam, Apache Spark, or other data processing frameworks
	* Data orchestration tools like Apache Airflow or AWS Step Functions
	* Data storage solutions like Amazon S3, Google Cloud Storage, or Azure Blob Storage
4. **Security and Access Control**: Understand our security measures, including authentication

### Layer 1 vs Layer 2 — two different kinds of memory

It is easy to conflate these, so let us look at both directly from SQLite:

- **Session history** is the *raw transcript* of this conversation.
- **User memories** are *distilled facts* the model judged worth keeping long-term. They are not a copy of the chat; they are a summary that can be reused in a completely different session.

In [ ]:
# Inspect what Agno persisted -- the two layers side by side.

# Layer 1: raw session message history (this conversation).
print("=== Session history (Layer 1) ===")
try:
    history = tutor.get_chat_history(session_id=SESSION_ID)
except Exception as e:
    history = []
    print(f"  (no session history yet: {e})")
for msg in history:
    snippet = (msg.content or "").replace("\n", " ")[:80]
    print(f"  [{msg.role}] {snippet}")

# Layer 2: distilled long-term facts about the user (reusable across sessions).
print("\n=== Long-term user memories (Layer 2) ===")
memories = tutor.get_user_memories(user_id=USER_ID) or []
if memories:
    for m in memories:
        print(f"  - {m.memory}   (topics: {m.topics})")
else:
    print("  (none yet -- the model decides what is worth remembering; if empty,"
          " re-run after another turn or use the 70B model)")

=== Session history (Layer 1) ===
  [user] Hi! My name is Aanya and I just joined NimbusCloud as a data engineer.
  [assistant] ### Welcome to NimbusCloud, Aanya It's great to have you on board as a data engi
  [user] Based on my role, what should I learn first at NimbusCloud?
  [assistant] ### Prioritized Learning for Data Engineers at NimbusCloud As a data engineer at

=== Long-term user memories (Layer 2) ===
  - User's name is Aanya   (topics: ['name'])
  - The user wants to learn about their role at NimbusCloud   (topics: ['role', 'NimbusCloud', 'learning'])


### Persistence across a restart

This is the whole point of a database-backed agent. We **delete the agent object entirely** and build a brand-new one pointing at the *same* SQLite file, `session_id`, and `user_id`. Nothing is in process memory anymore: if it still knows Aanya, that knowledge came off disk.

In [ ]:
# Simulate a server restart: throw away the live agent and rebuild from disk.
del tutor

agent_db_reloaded = SqliteDb(db_file=DB_FILE)     # same file on disk
tutor_resumed = Agent(
    name="NimbusTutor",
    model=AgnoGroq(id="llama-3.3-70b-versatile"),
    db=agent_db_reloaded,
    description="You are a helpful AI tutor for the NimbusCloud onboarding program.",
    add_history_to_context=True,
    num_history_runs=5,
    enable_user_memories=True,
    markdown=True,
)

# Same session id + user id => the agent reloads the prior conversation and
# memories straight from SQLite.
r3 = tutor_resumed.run(
    "Hey, do you remember my name and what my role is?",
    user_id=USER_ID,
    session_id=SESSION_ID,
)
print(r3.content)

ERROR    Error calling Groq API: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust 
         your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code':         
         'tool_use_failed', 'failed_generation': '<function=add_memory{"memory": "The user asked about their name  
         and role.", "topics": ["name", "role", "inquiry"]}</function>\n'}}

ERROR    Non-retryable model provider error: {"error":{"message":"Failed to call a function. Please adjust your    
         prompt. See 'failed_generation' for more                                                                  
         details.","type":"invalid_request_error","code":"tool_use_failed","failed_generation":"\u003cfunction=add_
         memory{\"memory\": \"The user asked about their name and role.\", \"topics\": [\"name\", \"role\",        
         \"inquiry\"]}\u003c/function\u003e\n"}}

WARNING  Error in memory creation: {"error":{"message":"Failed to call a function. Please adjust your prompt. See  
         'failed_generation' for more                                                                              
         details.","type":"invalid_request_error","code":"tool_use_failed","failed_generation":"\u003cfunction=add_
         memory{\"memory\": \"The user asked about their name and role.\", \"topics\": [\"name\", \"role\",        
         \"inquiry\"]}\u003c/function\u003e\n"}}

### Your Profile, Aanya
Yes, I remember that your name is **Aanya**, and you've joined NimbusCloud as a **Data Engineer**. I'm here to support you throughout the onboarding process and help you succeed in your role. If you have any questions or need assistance, feel free to ask!


In [ ]:
# And the long-term memories are still there too -- loaded from SQLite, not
# from anything this fresh agent object did in-process.
print("Memories available to the resumed agent:")
for m in (tutor_resumed.get_user_memories(user_id=USER_ID) or []):
    print(f"  - {m.memory}")

Memories available to the resumed agent:
  - User's name is Aanya
  - The user wants to learn about their role at NimbusCloud


**[Instructor, 1:48]**

That last cell is the magic moment. We threw away the agent object, created a brand new one, but pointed it at the **same `session_id`** in the same SQLite DB. The conversation history was rehydrated automatically. Aanya can come back tomorrow, next week, next quarter, and the agent picks up where it left off.

**In production**, you would swap `SqliteDb` for one of Agno's other DB backends (e.g. `PostgresDb`), and key sessions on the user's auth ID via `user_id` / `session_id`. That is essentially how Cursor, ChatGPT, and Claude maintain per-user memory.

**Chat prompt:** Type in chat one feature in your app where persistent agent memory would be a game-changer.

## Segment 8: LlamaIndex vs LangChain, MCQ 5, and Wrap-up (1:50 - 2:00)

**[Instructor, 1:50]**

Final segment. The question every team eventually asks: **"Should we use LlamaIndex or LangChain?"**.

**The iOS vs Android analogy.**

- **iOS (LlamaIndex):** opinionated, batteries-included, beautiful defaults for the most common job (data + retrieval). You move fast if you stay on the happy path. You hit walls when you want to do something exotic.
- **Android (LangChain):** more permissive, more wrappers, more flexibility, more rope to hang yourself with. Better when your workflow is a complex multi-step orchestration with many tools and branching logic.

Both can call the same LLMs. Both can connect to the same vector DBs. The difference is the **center of gravity**.

### Decision matrix

| If your project is mostly... | Lean towards |
|---|---|
| RAG over a large, varied corpus | **LlamaIndex** |
| Summarization of long documents | **LlamaIndex** |
| Knowledge graph or hybrid index needs | **LlamaIndex** |
| Multi-tool agentic workflows with branching logic | **LangChain** (or LangGraph) |
| You need many third-party integrations and chains | **LangChain** |
| You need persistent agent memory and team-of-agents patterns | **Agno** (often used alongside either) |
| Production RAG with strict observability | Either, with **LangSmith** or **LlamaIndex callbacks** |

**Honest take:** Most mature production teams use **both**. LlamaIndex for the data + retrieval layer, LangChain or LangGraph for orchestration, Agno for agent memory. They are not enemies. They are specialists.

### MCQ 5 of 5

**Question:** Your team is building a customer support copilot. The system needs to:
(1) retrieve answers from 200,000 support tickets and 50 product manuals,
(2) decide whether to call a refund API, a status API, or escalate to a human,
(3) remember each customer's past conversations across sessions.

Which stack is the MOST sensible split of responsibilities?

- **A.** LangChain for everything.
- **B.** LlamaIndex for everything.
- **C.** LlamaIndex for the retrieval layer over tickets and manuals, LangChain or LangGraph for tool routing and branching, Agno for per-customer persistent memory.
- **D.** Skip frameworks entirely and call the LLM API directly.

*React with A, B, C, or D.*

**Correct answer: C.** Each framework is used where it is strongest. LlamaIndex owns the data layer, LangChain/LangGraph owns the agentic orchestration, Agno owns the memory. This is the pattern you see in most production deployments today.

## Wrap-up (1:58 - 2:00)

**[Instructor, 1:58]**

Let us land the plane. Today you learned:

1. **Vector Index** for semantic Q&A (Netflix).
2. **Tree Index** for summarization (Wikipedia TOC).
3. **Keyword Index** for exact-term lookup (book back-index).
4. **Knowledge Graph Index** for relationship queries (Google Knowledge Panel).
5. **Sub-Question Query Engine** for complex multi-source questions (Amazon filters).
6. **Agno** for persistent agent memory (ChatGPT memory).
7. **When to choose LlamaIndex vs LangChain vs both** (iOS vs Android).